# QuDET Kickstart Guide

This notebook walks you through a complete end-to-end workflow using all six modules of the QuDET library. We will load a real-world dataset, preprocess it, encode features into quantum states, train quantum ML models, optimize circuits, and apply governance controls, all with QuDET.

---

## What you will learn

| # | Section | Module | What it does |
|---|---------|--------|--------------|
| 1 | Data Loading & Validation | **Connectors** | Load Parquet data, validate integrity, split into train/val/test |
| 2 | Feature Scaling & PCA | **Transforms** | Standardize features, reduce dimensionality with Quantum PCA |
| 3 | Outlier Detection | **Transforms** | Remove anomalous samples with Z-score method |
| 4 | Quantum Encoding | **Encoders** | Map classical features to quantum circuits (Angle, Rotation, Amplitude) |
| 5 | Classification & Clustering | **Analytics** | Train Quantum SVC and Quantum KMeans on encoded data |
| 6 | Circuit Optimization | **Compute** | Reduce circuit depth and gate count before execution |
| 7 | Drift Detection & Encryption | **Governance** | Monitor data drift, encrypt secrets, audit operations |
| 8 | Model Serialization | **Connectors** | Save and reload trained models for production |

---

### Prerequisites

```bash
pip install qudet
```

### Dataset

We use the **UCI Wine Recognition** dataset (178 samples, 13 chemical features, 3 cultivar classes) stored as a Parquet file in `dataset/wine_quality.parquet`.

---

## 1. Data Loading & Exploration with `qudet.connectors`

QuDET's connectors module handles data ingestion from CSV, Parquet, and SQL sources. Let's load the wine dataset and inspect it.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Load the Wine Quality dataset from parquet
df = pd.read_parquet("dataset/wine_quality.parquet")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (178, 14)
Columns: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline', 'target']


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [2]:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"].values

print(f"Features: {X.shape[1]}")
print(f"Samples:  {len(y)}")
print(f"\nClass distribution:")
print(df["target"].value_counts().sort_index())

Features: 13
Samples:  178

Class distribution:
target
0    59
1    71
2    48
Name: count, dtype: int64


### 1.1 Data Validation

Before feeding data into a quantum pipeline, we should validate that it is clean with no `NaN`, no `Inf`, and proper shape. QuDET provides `DataValidator` for this.

In [3]:
from qudet.connectors.streaming import DataValidator

validator = DataValidator(allow_nan=False)
is_valid = validator.validate(X)

print(f"Data passes validation: {is_valid}")
print(f"Validation errors:      {validator.validation_errors}")

Data passes validation: True
Validation errors:      []


### 1.2 Train / Validation / Test Split

`DataSplitter` provides reproducible data splitting with configurable ratios.

In [4]:
from qudet.connectors.utilities import DataSplitter

splitter = DataSplitter(random_state=42)

# Split features
splits = splitter.split(X.values, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)
X_train, X_val, X_test = splits["train"], splits["val"], splits["test"]

# Split labels with the same seed
y_splits = splitter.split(y, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)
y_train, y_val, y_test = y_splits["train"], y_splits["val"], y_splits["test"]

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")

Train: (124, 13)  |  Val: (26, 13)  |  Test: (28, 13)


---

## 2. Feature Engineering with `qudet.transforms`

Raw features often have different scales. Quantum circuits are sensitive to input magnitudes, so proper normalization is critical.

### 2.1 Feature Scaling

QuDET's `FeatureScaler` supports `"standard"` (zero-mean, unit-variance) and `"minmax"` scaling. It follows scikit-learn's `fit` / `transform` pattern.

In [5]:
from qudet.transforms.feature_engineering import FeatureScaler

scaler = FeatureScaler(method="standard")
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Before scaling:")
print(f"  Mean (first 3 features): {X_train.mean(axis=0)[:3].round(2)}")
print(f"  Std  (first 3 features): {X_train.std(axis=0)[:3].round(2)}")
print("\nAfter scaling:")
print(f"  Mean (first 3 features): {X_train_scaled.mean(axis=0)[:3].round(2)}")
print(f"  Std  (first 3 features): {X_train_scaled.std(axis=0)[:3].round(2)}")

Before scaling:
  Mean (first 3 features): [12.95  2.29  2.35]
  Std  (first 3 features): [0.82 1.11 0.28]

After scaling:
  Mean (first 3 features): [0. 0. 0.]
  Std  (first 3 features): [1. 1. 1.]


### 2.2 Quantum PCA with Dimensionality Reduction

`QuantumPCA` uses a quantum kernel matrix to perform kernel PCA. This is especially useful for capturing non-linear relationships that standard PCA misses.

We reduce from 13 features down to 4, making the data manageable for quantum circuits.

In [6]:
from qudet.transforms.pca import QuantumPCA

pca = QuantumPCA(n_components=4)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original dimensions: {X_train_scaled.shape[1]}")
print(f"Reduced dimensions:  {X_train_pca.shape[1]}")
print(f"Reduced data shape:  {X_train_pca.shape}")

Original dimensions: 13
Reduced dimensions:  4
Reduced data shape:  (124, 4)


### 2.3 Outlier Removal

`OutlierRemover` detects and removes outliers using either IQR or Z-score methods. This is important because quantum circuits can be sensitive to extreme values.

In [7]:
from qudet.transforms.feature_engineering import OutlierRemover

remover = OutlierRemover(method="zscore", threshold=3.0)
X_train_clean = remover.fit_transform(X_train_pca)
outlier_ratio = remover.get_outlier_ratio(X_train_pca)

print(f"Before: {X_train_pca.shape[0]} samples")
print(f"After:  {X_train_clean.shape[0]} samples")
print(f"Outlier ratio: {outlier_ratio:.2%}")

Before: 124 samples
After:  124 samples
Outlier ratio: 0.00%


---

## 3. Quantum Encoding with `qudet.encoders`

Classical data must be encoded into quantum states before quantum processing. QuDET provides multiple encoding strategies, each with different trade-offs:

| Encoder | Strategy | Qubits needed | Best for |
|---------|----------|---------------|----------|
| `AngleEncoder` | One rotation gate per feature | n_features | Preserving individual feature values |
| `RotationEncoder` | Parameterized rotations | n_features | General-purpose encoding |
| `AmplitudeEncoder` | Encode into state amplitudes | log2(n_features) | Compact representation |

Let's encode a single sample (4 PCA features) using all three approaches.

In [8]:
from qudet.encoders import AngleEncoder, AmplitudeEncoder, RotationEncoder

sample = X_train_pca[0]
print(f"Sample to encode (4 features): {sample.round(3)}")

# --- Angle Encoding (Ry rotations) ---
angle_enc = AngleEncoder(n_qubits=4, angle_type="ry")
qc_angle = angle_enc.encode(sample)
print(f"\nAngle Encoder (Ry):  qubits={qc_angle.num_qubits}, depth={qc_angle.depth()}")

# --- Rotation Encoding ---
rot_enc = RotationEncoder(n_qubits=4)
qc_rot = rot_enc.encode(sample)
print(f"Rotation Encoder:    qubits={qc_rot.num_qubits}, depth={qc_rot.depth()}")

# --- Amplitude Encoding (log2 compression) ---
amp_enc = AmplitudeEncoder(n_qubits=2)
qc_amp = amp_enc.encode(sample)
print(f"Amplitude Encoder:   qubits={qc_amp.num_qubits}, depth={qc_amp.depth()}")

Sample to encode (4 features): [-0.022  0.603 -0.15   0.227]

Angle Encoder (Ry):  qubits=4, depth=1
Rotation Encoder:    qubits=4, depth=1
Amplitude Encoder:   qubits=2, depth=1


In [9]:
# Let's visualize the angle-encoded circuit
print("Angle-encoded circuit for one wine sample:")
print(qc_angle.draw())

Angle-encoded circuit for one wine sample:


     ┌───────────────┐
q_0: ┤ Ry(-0.021959) ├
     └┬─────────────┬┘
q_1: ─┤ Ry(0.60323) ├─
      ├─────────────┴┐
q_2: ─┤ Ry(-0.15015) ├
      ├─────────────┬┘
q_3: ─┤ Ry(0.22687) ├─
      └─────────────┘ 


---

## 4. Quantum Machine Learning with `qudet.analytics`

QuDET provides quantum-enhanced ML models that use quantum kernel methods under the hood. They follow scikit-learn conventions (`fit`, `predict`, `score`).

> **Note:** Quantum kernel computation is O(n²) in the number of samples. For this demo, we use small subsets to keep execution fast.

### 4.1 Quantum Support Vector Classifier

In [10]:
from qudet.analytics.classifier import QuantumSVC

# Use a small subset (quantum kernel is O(n^2))
np.random.seed(42)
small_idx = np.random.choice(len(X_train_pca), size=30, replace=False)
X_small = X_train_pca[small_idx]
y_small = y_train[small_idx]

# Binary classification: class 0 vs rest
y_binary = (y_small == 0).astype(int)
X_small_2d = X_small[:, :2]  # Use 2 PCA components for speed

# Train
svc = QuantumSVC(n_qubits=2)
svc.fit(X_small_2d, y_binary)
train_acc = svc.score(X_small_2d, y_binary)

print(f"Quantum SVC — Binary Classification")
print(f"  Training samples:  {len(X_small_2d)}")
print(f"  Training accuracy: {train_acc:.2%}")

Quantum SVC — Binary Classification
  Training samples:  30
  Training accuracy: 63.33%


### 4.2 Quantum KMeans Clustering

`QuantumKMeans` uses quantum kernel distances to assign cluster membership. No labels needed, this is unsupervised learning.

In [11]:
from qudet.analytics.clustering import QuantumKMeans

X_cluster = X_train_pca[:25, :2]

kmeans = QuantumKMeans(n_clusters=3, n_qubits=2)
kmeans.fit(X_cluster)
labels = kmeans.predict(X_cluster)

print(f"Quantum KMeans — Unsupervised Clustering")
print(f"  Samples:  {len(X_cluster)}")
print(f"  Clusters: {np.unique(labels)}")
print(f"  Distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")

Quantum KMeans — Unsupervised Clustering
  Samples:  25
  Clusters: [0 1 2]
  Distribution: {np.int64(0): np.int64(8), np.int64(1): np.int64(5), np.int64(2): np.int64(12)}


---

## 5. Circuit Optimization with `qudet.compute`

Real quantum hardware has limited coherence time. Every unnecessary gate adds noise. QuDET's `CircuitOptimizer` automatically simplifies circuits by removing redundant gates and applying Qiskit transpiler passes.

### 5.1 Optimize a Redundant Circuit

In [12]:
from qiskit import QuantumCircuit
from qudet.compute.simplify import CircuitOptimizer

# Build a circuit with redundant gates
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.h(0)
qc.h(0)  # This H cancels with the previous one

print(f"Original — depth: {qc.depth()}, gates: {qc.size()}")
print(qc.draw())

# Optimize
optimizer = CircuitOptimizer(level=3)
qc_opt = optimizer.optimize(qc)

print(f"\nOptimized — depth: {qc_opt.depth()}, gates: {qc_opt.size()}")
print(qc_opt.draw())

Original — depth: 4, gates: 5
     ┌───┐     ┌───┐┌───┐
q_0: ┤ H ├──■──┤ H ├┤ H ├
     └───┘┌─┴─┐└───┘└───┘
q_1: ─────┤ X ├──■───────
          └───┘┌─┴─┐     
q_2: ──────────┤ X ├─────
               └───┘     



Optimized — depth: 3, gates: 3
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘


### 5.2 Backend Management

`BackendManager` provides a unified interface to discover and select quantum backends (simulators or real hardware).

In [13]:
from qudet.compute.backend import BackendManager

manager = BackendManager()
backend = manager.get_backend("aer_simulator")
print(f"Selected backend: {backend.name}")

Connection failed: 'channel' can only be 'ibm_cloud', or 'ibm_quantum_platform — falling back to simulator


Selected backend: aer_simulator


---

## 6. Governance & Safety with `qudet.governance`

Production quantum pipelines need monitoring, security, and auditability. QuDET's governance module provides these controls.

### 6.1 Quantum Drift Detection

`QuantumDriftDetector` monitors whether your incoming data distribution has shifted compared to the reference data your model was trained on. It uses the **Maximum Mean Discrepancy (MMD)** statistic computed via quantum kernels.

In [14]:
from qudet.governance.drift import QuantumDriftDetector

# Reference data (what the model was trained on)
X_ref = X_train_pca[:40, :2]

# Simulate a distribution shift
X_shifted = X_ref + np.random.normal(0.5, 0.1, X_ref.shape)

detector = QuantumDriftDetector(n_qubits=2, threshold=0.1)
detector.fit_reference(X_ref)

# Test with same distribution (should NOT detect drift)
result_ok = detector.detect_drift(X_ref)
print(f"Same distribution:")
print(f"  Drift detected: {result_ok['drift_detected']}")
print(f"  MMD score:      {result_ok['mmd_score']:.4f}")

# Test with shifted distribution (should detect drift)
result_drift = detector.detect_drift(X_shifted)
print(f"\nShifted distribution:")
print(f"  Drift detected: {result_drift['drift_detected']}")
print(f"  MMD score:      {result_drift['mmd_score']:.4f}")

Same distribution:
  Drift detected: False
  MMD score:      0.0000



Shifted distribution:
  Drift detected: True
  MMD score:      0.1928


### 6.2 Data Encryption

`EncryptionManager` provides Fernet symmetric encryption (AES-128-CBC + HMAC-SHA256) for protecting sensitive pipeline metadata, model parameters, or dataset credentials.

In [15]:
from qudet.governance.security import EncryptionManager

em = EncryptionManager()
em.generate_key("pipeline_key")

# Encrypt sensitive metadata
secret = "model_accuracy=0.95;dataset=wine;version=1.0"
success, data_id = em.encrypt_data(secret, "pipeline_key")
print(f"Encrypted: {success}")

# Decrypt it back
success, plaintext = em.decrypt_data(data_id, "pipeline_key")
print(f"Decrypted: {plaintext}")
print(f"Match:     {plaintext == secret}")

Encrypted: True
Decrypted: model_accuracy=0.95;dataset=wine;version=1.0
Match:     True


### 6.3 Audit Logging

`AuditLogger` records all pipeline operations with timestamps, users, and metadata which is essential for compliance and debugging.

In [16]:
from qudet.governance.audit import AuditLogger

audit = AuditLogger()

# Log the training event
audit.log_event(
    event_type="model_training",
    user="data_scientist",
    action="train_quantum_svc",
    resource="wine_dataset",
    status="success",
    details={"accuracy": train_acc, "n_qubits": 2, "n_samples": 30}
)

print(f"Events logged: {len(audit.events)}")
event = audit.events[-1]
print(f"Latest: [{event.event_type}] {event.action} on {event.resource} — {event.status}")

Events logged: 1
Latest: [model_training] train_quantum_svc on wine_dataset — success


---

## 7. Model Serialization with `qudet.connectors`

Save trained models to disk and reload them later using `QuantumSerializer`. This enables deploying trained quantum pipelines to production.

In [17]:
import os
from qudet.connectors.serialization import QuantumSerializer

# Save the trained scaler
QuantumSerializer.save_model(scaler, "dataset/trained_scaler.pkl")

# Load it back
loaded = QuantumSerializer.load_model("dataset/trained_scaler.pkl")
print(f"Saved and loaded FeatureScaler")
print(f"Loaded scaler method: {loaded.method}")

# Verify it produces the same output
test_result = loaded.transform(X_test)
print(f"Loaded scaler output shape: {test_result.shape}")

# Cleanup
os.remove("dataset/trained_scaler.pkl")

Saved and loaded FeatureScaler
Loaded scaler method: standard
Loaded scaler output shape: (28, 13)
